In [5]:
import sys
import os
import numpy as np
import xarray as xr

from pyPAS.model.material import Material
from pyPAS.model.layer import Layer
from pyPAS.model.sample import Sample
from pyPAS.transport.diffusion.positron_profile_solver import profile_solver
from pyPAS.analysis.vedb.annihilation_fractions import compute_annihilation_fractions

In [ ]:
from ../scipy_positron_profile_solver import scipy_profile_solver


In [ ]:
surface = Material(name="surface", diffusion=0.10, mobility=0.0, bulk_annihilation_rate=0.01)
    bulk    = Material(name="bulk",    diffusion=0.10, mobility=0.0, bulk_annihilation_rate=0.001)
    return Sample(
        layers=[Layer(material=surface, width=20.0), Layer(material=bulk, width=280.0)],
        absorption_length=2.0,
    )

In [ ]:



sample = two_layer_sample
source = _gaussian_source(sample.sample_length(), center=50.0, sigma=8.0)
fd = profile_solver(source, sample, mesh_size=3000)
sc = scipy_profile_solver(source, sample, num_of_mesh_cells=1000, initial_guess=fd, max_nodes=5000)
# scipy's collocation residual check stalls at the λ-discontinuity (z=20 nm)
# and does not formally converge, but because the FD initial guess is already
# near the true solution the iterates are physically correct.  We use sc.sol
# regardless of sc.success, matching the notebook's approach.
x = fd.coords["x"].values
sc_profile = xr.DataArray(np.clip(sc.sol(x)[0], 0.0, None), coords={"x": x})
fd_fracs = compute_annihilation_fractions(fd, sample)
sc_fracs = compute_annihilation_fractions(sc_profile, sample)
for layer_idx in fd_fracs.coords["layer"].values:
    fd_f = float(fd_fracs.sel(layer=layer_idx))
    sc_f = float(sc_fracs.sel(layer=layer_idx))
    assert abs(fd_f - sc_f) < 0.01, (
        f"Layer {layer_idx}: annihilation fractions differ by {abs(fd_f - sc_f):.4f} "
        f"(FD={fd_f:.4f}, scipy={sc_f:.4f})"
    )
